In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS iot_sensor_anomoly_detection.bronze;

In [0]:
#final code for bronze layer

In [0]:
# --------------------------------------------
# Bronze ingestion for IoT Sensor datasets
# --------------------------------------------
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_iot_pipeline")

# --------------------------------------------
# Dataset configuration
# --------------------------------------------
datasets = {
    "device_health_diagnostics": "s3://iot-sensor-target/2026/03/11/17/",
    "device_operations": "s3://iot-sensor-target/2026/03/11/19/",
    "environment_network": "s3://iot-sensor-target/2026/03/11/20/",
    "sensor_stream": "s3://iot-sensor-target/2026/03/11/21/",
    "time_anomaly_events": "s3://iot-sensor-target/2026/03/11/22/"
}

BRONZE_SCHEMA = "iot_sensor_anomoly_detection.bronze"

# --------------------------------------------
# Ensure Bronze schema exists
# --------------------------------------------
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")

# --------------------------------------------
# Bronze ingestion pipeline
# --------------------------------------------
for table_name, path in datasets.items():

    full_table_name = f"{BRONZE_SCHEMA}.{table_name}"

    try:
        logger.info(f"Starting ingestion for {table_name}")
        logger.info(f"Source path: {path}")

        # ------------------------------------------------
        # Check if S3 path exists
        # ------------------------------------------------
        try:
            files = dbutils.fs.ls(path)
        except Exception:
            logger.warning(f"Source path does not exist: {path}")
            files = []

        if len(files) == 0:
            logger.warning(f"No files found at {path}. Skipping ingestion.")
            continue

        # ------------------------------------------------
        # Read Parquet files
        # ------------------------------------------------
        df = spark.read.format("parquet").load(path)

        # Normalize column names
        df = df.toDF(*[c.lower() for c in df.columns])

        # ------------------------------------------------
        # Write Bronze table as Delta
        # ------------------------------------------------
        df.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .saveAsTable(full_table_name)

        logger.info(f"Bronze table created successfully: {full_table_name}")

    except Exception as e:
        logger.error(f"Ingestion failed for {table_name}")
        logger.error(str(e))

logger.info("Bronze ingestion pipeline completed")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.bronze.device_health_diagnostics
LIMIT 20;

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.bronze.environment_network
LIMIT 20;

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.bronze.sensor_stream
LIMIT 20;

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.bronze.time_anomaly_events
LIMIT 20;

In [0]:
%sql
SELECT 
COUNT(*) - COUNT(DISTINCT device_id) AS duplicate_count
FROM iot_sensor_anomoly_detection.bronze.device_health_diagnostics;

In [0]:
from pyspark.sql.functions import when, rand, col

# Read Bronze table
df = spark.table("iot_sensor_anomoly_detection.bronze.device_health_diagnostics")

# Introduce NULL values randomly
df_with_nulls = df \
.withColumn(
    "health_score",
    when(rand() < 0.2, None).otherwise(col("health_score"))
) \
.withColumn(
    "battery_voltage",
    when(rand() < 0.2, None).otherwise(col("battery_voltage"))
) \
.withColumn(
    "maintenance_cost",
    when(rand() < 0.2, None).otherwise(col("maintenance_cost"))
) \
.withColumn(
    "support_team",
    when(rand() < 0.2, None).otherwise(col("support_team"))
)

# Overwrite Bronze table
df_with_nulls.write.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("iot_sensor_anomoly_detection.bronze.device_health_diagnostics")

print("✅ Random NULL values added successfully")

In [0]:
%sql
SELECT
SUM(CASE WHEN health_score IS NULL THEN 1 ELSE 0 END) AS null_health_score,
SUM(CASE WHEN battery_voltage IS NULL THEN 1 ELSE 0 END) AS null_battery_voltage,
SUM(CASE WHEN maintenance_cost IS NULL THEN 1 ELSE 0 END) AS null_maintenance_cost,
SUM(CASE WHEN support_team IS NULL THEN 1 ELSE 0 END) AS null_support_team
FROM iot_sensor_anomoly_detection.bronze.device_health_diagnostics;

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.bronze.device_health_diagnostics
LIMIT 100;

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.bronze.environment_network
LIMIT 100;

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.bronze.sensor_stream
LIMIT 100;

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.bronze.time_anomaly_events
LIMIT 100;

Table 2 Device operations table 

In [0]:
# --------------------------------------------
# Bronze ingestion for device_operations
# --------------------------------------------
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_device_operations")

BRONZE_SCHEMA = "iot_sensor_anomoly_detection.bronze"
SOURCE_PATH = "s3://iot-sensor-target/2026/03/11/19/"
TABLE_NAME = f"{BRONZE_SCHEMA}.device_operations"

try:

    logger.info("Starting ingestion for device_operations")

    # Ensure Bronze schema exists
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")

    # Check if files exist
    try:
        files = dbutils.fs.ls(SOURCE_PATH)
    except Exception:
        logger.warning(f"Source path does not exist: {SOURCE_PATH}")
        files = []

    if len(files) == 0:
        logger.warning("No files found in source path.")
    else:

        # Read Parquet files
        df = spark.read.format("parquet").load(SOURCE_PATH)

        # Normalize column names
        df = df.toDF(*[c.lower() for c in df.columns])

        # Write as Delta table
        df.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .saveAsTable(TABLE_NAME)

        logger.info(f"Delta table created successfully: {TABLE_NAME}")

except Exception as e:
    logger.error("Ingestion failed for device_operations")
    logger.error(str(e))

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.bronze.device_operations
LIMIT 100;

In [0]:
%sql
SELECT 
COUNT(*) - COUNT(DISTINCT device_id) AS duplicate_count
FROM iot_sensor_anomoly_detection.bronze.device_operations;

In [0]:
%sql
SELECT 
COUNT(*) - COUNT(DISTINCT device_id) AS duplicate_count
FROM iot_sensor_anomoly_detection.bronze.environment_network;

In [0]:
%sql
SELECT 
COUNT(*) - COUNT(DISTINCT device_id) AS duplicate_count
FROM iot_sensor_anomoly_detection.bronze.sensor_stream;

In [0]:
%sql
SELECT 
COUNT(*) - COUNT(DISTINCT device_id) AS duplicate_count
FROM iot_sensor_anomoly_detection.bronze.time_anomaly_events;

In [0]:
from pyspark.sql.functions import col, rand, when

table_name = "iot_sensor_anomoly_detection.bronze.device_operations"

df = spark.table(table_name)

df_with_nulls = df \
.withColumn(
    "maker_name",
    when(rand() < 0.1, None).otherwise(col("maker_name"))
) \
.withColumn(
    "factory_site",
    when(rand() < 0.1, None).otherwise(col("factory_site"))
) \
.withColumn(
    "battery_pct",
    when(rand() < 0.1, None).otherwise(col("battery_pct"))
) \
.withColumn(
    "cpu_load",
    when(rand() < 0.1, None).otherwise(col("cpu_load"))
) \
.withColumn(
    "memory_usage",
    when(rand() < 0.1, None).otherwise(col("memory_usage"))
) \
.withColumn(
    "device_temp",
    when(rand() < 0.1, None).otherwise(col("device_temp"))
)

df_with_nulls.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(table_name)

print("Random NULL values inserted successfully")

In [0]:
%sql
SELECT
SUM(CASE WHEN maker_name IS NULL THEN 1 ELSE 0 END) AS null_maker_name,
SUM(CASE WHEN factory_site IS NULL THEN 1 ELSE 0 END) AS null_factory_site,
SUM(CASE WHEN battery_pct IS NULL THEN 1 ELSE 0 END) AS null_battery,
SUM(CASE WHEN cpu_load IS NULL THEN 1 ELSE 0 END) AS null_cpu,
SUM(CASE WHEN memory_usage IS NULL THEN 1 ELSE 0 END) AS null_memory,
SUM(CASE WHEN device_temp IS NULL THEN 1 ELSE 0 END) AS null_temp
FROM iot_sensor_anomoly_detection.bronze.device_operations;

In [0]:
from pyspark.sql.functions import col, rand, when

table_name = "iot_sensor_anomoly_detection.bronze.environment_network"

df = spark.table(table_name)

df_with_nulls = df \
.withColumn(
    "region_name",
    when(rand() < 0.1, None).otherwise(col("region_name"))
) \
.withColumn(
    "network_provider",
    when(rand() < 0.1, None).otherwise(col("network_provider"))
) \
.withColumn(
    "ambient_temp",
    when(rand() < 0.1, None).otherwise(col("ambient_temp"))
) \
.withColumn(
    "humidity_pct",
    when(rand() < 0.1, None).otherwise(col("humidity_pct"))
) \
.withColumn(
    "network_latency",
    when(rand() < 0.1, None).otherwise(col("network_latency"))
)

df_with_nulls.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(table_name)

print("Random NULL values inserted successfully")

In [0]:
%sql
SELECT
SUM(CASE WHEN region_name IS NULL THEN 1 ELSE 0 END) AS null_region,
SUM(CASE WHEN network_provider IS NULL THEN 1 ELSE 0 END) AS null_provider,
SUM(CASE WHEN ambient_temp IS NULL THEN 1 ELSE 0 END) AS null_temp,
SUM(CASE WHEN humidity_pct IS NULL THEN 1 ELSE 0 END) AS null_humidity,
SUM(CASE WHEN network_latency IS NULL THEN 1 ELSE 0 END) AS null_latency
FROM iot_sensor_anomoly_detection.bronze.environment_network;

In [0]:
from pyspark.sql.functions import col, rand, when

table_name = "iot_sensor_anomoly_detection.bronze.sensor_stream"

df = spark.table(table_name)

df_with_nulls = df \
.withColumn(
    "sensor_category",
    when(rand() < 0.1, None).otherwise(col("sensor_category"))
) \
.withColumn(
    "reading_unit",
    when(rand() < 0.1, None).otherwise(col("reading_unit"))
) \
.withColumn(
    "reading_value",
    when(rand() < 0.1, None).otherwise(col("reading_value"))
) \
.withColumn(
    "signal_strength",
    when(rand() < 0.1, None).otherwise(col("signal_strength"))
) \
.withColumn(
    "noise_level",
    when(rand() < 0.1, None).otherwise(col("noise_level"))
)

df_with_nulls.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(table_name)

print("Random NULL values inserted successfully into sensor_stream Bronze table")

In [0]:
%sql
SELECT
SUM(CASE WHEN sensor_category IS NULL THEN 1 ELSE 0 END) AS null_category,
SUM(CASE WHEN reading_unit IS NULL THEN 1 ELSE 0 END) AS null_unit,
SUM(CASE WHEN reading_value IS NULL THEN 1 ELSE 0 END) AS null_reading,
SUM(CASE WHEN signal_strength IS NULL THEN 1 ELSE 0 END) AS null_signal,
SUM(CASE WHEN noise_level IS NULL THEN 1 ELSE 0 END) AS null_noise
FROM iot_sensor_anomoly_detection.bronze.sensor_stream;

In [0]:
from pyspark.sql.functions import col, rand, when

table_name = "iot_sensor_anomoly_detection.bronze.time_anomaly_events"

df = spark.table(table_name)

df_with_nulls = df \
.withColumn(
    "event_type",
    when(rand() < 0.1, None).otherwise(col("event_type"))
) \
.withColumn(
    "anomaly_category",
    when(rand() < 0.1, None).otherwise(col("anomaly_category"))
) \
.withColumn(
    "root_cause_hint",
    when(rand() < 0.1, None).otherwise(col("root_cause_hint"))
) \
.withColumn(
    "downtime_minutes",
    when(rand() < 0.1, None).otherwise(col("downtime_minutes"))
) \
.withColumn(
    "event_score",
    when(rand() < 0.1, None).otherwise(col("event_score"))
)

df_with_nulls.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(table_name)

print("Random NULL values inserted successfully into time_anomaly_events Bronze table")

In [0]:
%sql
SELECT
SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) AS null_event_type,
SUM(CASE WHEN anomaly_category IS NULL THEN 1 ELSE 0 END) AS null_anomaly,
SUM(CASE WHEN root_cause_hint IS NULL THEN 1 ELSE 0 END) AS null_root_cause,
SUM(CASE WHEN downtime_minutes IS NULL THEN 1 ELSE 0 END) AS null_downtime,
SUM(CASE WHEN event_score IS NULL THEN 1 ELSE 0 END) AS null_score
FROM iot_sensor_anomoly_detection.bronze.time_anomaly_events;